# 실습 - SAM 미세조정 (세그멘테이션)

**의료 영상에 맞게 SAM을 조정**

**SAM의 특성**

- 프롬프트 기반 — 점 · 박스로 대상을 지정하여 분할
- 사전학습 규모 — 이미지 1,100만 장, 마스크 10억 개
- 범용성 — 범주를 학습하지 않아 어떤 영상에도 적용
- 한계 — 의료 영상은 학습에 포함되지 않음

**학습 범위**

| 구성 | 파라미터 | 처리 |
|---|---|---|
| 비전 인코더 | 89.7M | **동결** |
| 프롬프트 인코더 | 6.2K | **동결** |
| 마스크 디코더 | 4.1M | **학습** |

- 학습 대상 — 전체의 약 4.3%
- 앞서 다룬 **부분 미세조정** 방식

**데이터 — Kvasir-SEG**

- 대장내시경 용종 이미지 1,000장
- 의료 전문가가 픽셀 단위로 주석
- 용종 크기 편차 큼 — 화면의 1%에서 50%까지

**실습 순서**

- 환경 준비 · 데이터 불러오기
- 전처리 — 마스크에서 박스 프롬프트 추출
- 모델 구성 — 인코더 동결
- 학습 실행
- 미세조정 전후 비교

---
## 0. 환경 준비

- `transformers` — SAM 모델과 처리기
- `datasets` — Hub 데이터셋
- `torch` — 학습 수행
- 별도 라이브러리 설치 불필요

In [ ]:
from datasets import load_dataset
from transformers import SamModel, SamProcessor
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import random, time

device = "cuda" if torch.cuda.is_available() else "cpu"

print("준비 완료")
print("장치:", device)

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- 대장내시경 용종 이미지와 정답 마스크
- `N_TRAIN` — 학습에 사용할 이미지 수
- 많을수록 성능은 오르나 학습 시간 증가

In [ ]:
N_TRAIN = 500
N_EVAL = 100

ds_all = load_dataset("kowndinya23/Kvasir-SEG", split=f"train[:{N_TRAIN + N_EVAL}]")

split = ds_all.train_test_split(test_size=N_EVAL, seed=42)
ds = split["train"]        # 학습용
ds_test = split["test"]    # 평가용 — 학습에 사용하지 않음

print(f"학습 {len(ds)}장 / 평가 {len(ds_test)}장")

### 1-2. 이미지와 마스크 확인

- `image` — 내시경 촬영 이미지
- `annotation` — 정답 마스크 (용종 영역)
- 마스크는 흑백 — 대상은 흰색, 배경은 검은색

In [ ]:
item = ds[0]

for k, v in item.items():
    print(f"{k}: {str(v)[:100]}")

item["image"]

In [ ]:
# 정답 마스크 확인 (열 이름은 위 출력에 맞게 수정)
MASK_COL = "annotation"

item[MASK_COL]

---
## 2. 전처리

### 2-1. 박스 프롬프트 추출

- SAM은 **프롬프트**를 받아 분할 — 점 · 박스 · 마스크
- 학습 시 정답 마스크에서 박스를 자동 추출
- `shift` — 박스에 무작위 여유를 주어 부정확한 입력에 대비

**학습과 평가의 차이**

- 학습 시 — `shift=20`, 흔들어서 강건하게
- 평가 시 — `shift=0`, 정확한 박스로 공정하게 비교

In [ ]:
def get_bounding_box(mask, shift=20):
    """정답 마스크에서 박스 프롬프트 추출"""
    arr = np.array(mask)
    if arr.ndim == 3:
        arr = arr[:, :, 0]

    y_idx, x_idx = np.where(arr > 0)
    if len(x_idx) == 0:
        h, w = arr.shape
        return [0, 0, w, h]

    x_min, x_max = int(x_idx.min()), int(x_idx.max())
    y_min, y_max = int(y_idx.min()), int(y_idx.max())
    H, W = arr.shape

    if shift > 0:
        x_min = max(0, x_min - np.random.randint(0, shift))
        x_max = min(W, x_max + np.random.randint(0, shift))
        y_min = max(0, y_min - np.random.randint(0, shift))
        y_max = min(H, y_max + np.random.randint(0, shift))

    return [x_min, y_min, x_max, y_max]


print("박스 예시:", get_bounding_box(ds[0][MASK_COL], shift=0))

### 2-2. 데이터셋 클래스 정의

- 이미지 → `SamProcessor` 로 변환 (1024×1024)
- 마스크 → 256×256 으로 축소 (SAM의 출력 해상도)
- 박스 → 프롬프트로 전달

**참고** — 분류 미세조정은 `with_transform` 한 줄로 끝났으나, SAM은 입력이 이미지·박스·마스크 세 가지라 별도 클래스가 필요하다.

In [ ]:
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")


class PolypDataset(Dataset):
    def __init__(self, dataset, processor, mask_col="annotation"):
        self.dataset = dataset
        self.processor = processor
        self.mask_col = mask_col

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"].convert("RGB")
        mask = item[self.mask_col]

        box = get_bounding_box(mask, shift=20)

        inputs = self.processor(
            image, input_boxes=[[box]], return_tensors="pt"
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        m = mask.convert("L").resize((256, 256))
        m = (np.array(m) > 0).astype(np.float32)
        inputs["ground_truth_mask"] = torch.from_numpy(m)

        return inputs


train_ds = PolypDataset(ds, processor, MASK_COL)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)

for k, v in train_ds[0].items():
    print(f"{k}: {tuple(v.shape)}")

---
## 3. 모델 구성

### 3-1. 모델 불러오기

- `facebook/sam-vit-base` — 추론 실습에서 사용한 것과 동일한 모델
- 이번에는 `pipeline` 이 아닌 **모델을 직접** 불러옴

In [ ]:
model = SamModel.from_pretrained("facebook/sam-vit-base").to(device)

for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"{name}: {n:,}")

### 3-2. 학습 범위 지정

- 비전 인코더 · 프롬프트 인코더 — **동결**
- 마스크 디코더 — **학습**
- 근거 — 인코더는 이미 풍부한 시각 특징을 보유, 도메인 적응은 디코더에서 발생

In [ ]:
for name, param in model.named_parameters():
    if name.startswith("vision_encoder") or name.startswith("prompt_encoder"):
        param.requires_grad_(False)

total = sum(p.numel() for p in model.parameters())
train_n = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total:,}")
print(f"학습 대상    : {train_n:,} ({train_n/total:.2%})")

---
## 4. 학습 설정

### 4-1. 손실 함수

- **BCE** — 픽셀마다 독립적으로 정답 여부 계산
- **Dice** — 예측과 정답이 겹치는 정도를 직접 최적화

**두 손실을 함께 쓰는 이유**

- 용종은 화면의 일부만 차지 — 배경 픽셀이 압도적으로 많음
- BCE만 사용하면 "대부분 배경"으로 기울어 경계가 뭉개짐
- Dice가 이 불균형을 보완 — 원 논문의 방식

In [ ]:
class DiceBCELoss(nn.Module):
    """Dice와 BCE를 결합한 손실 함수"""

    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.w = dice_weight

    def forward(self, logits, target):
        bce = self.bce(logits, target)

        prob = torch.sigmoid(logits)
        num = 2 * (prob * target).sum(dim=(1, 2, 3)) + 1e-6
        den = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1e-6
        dice = 1 - (num / den).mean()

        return (1 - self.w) * bce + self.w * dice


loss_fn = DiceBCELoss(dice_weight=0.5)
print("손실 함수 준비 완료")

### 4-2. 옵티마이저

- **대상** — 마스크 디코더만
- **학습률** `5e-5` — 짧은 학습에 맞추어 설정
- 에폭을 많이 돌릴 경우 `1e-5` 로 낮춤

In [ ]:
EPOCHS = 3
LR = 5e-5

optimizer = AdamW(model.mask_decoder.parameters(), lr=LR, weight_decay=0.01)

print(f"에폭 {EPOCHS} · 학습률 {LR}")
print(f"총 스텝 수: {len(train_loader) * EPOCHS}")

---
## 5. 학습 실행

- 입력이 이미지·박스·마스크 세 가지 — `Trainer` 대신 **학습 루프 직접 작성**
- `multimask_output=False` — 프롬프트당 마스크 하나만 생성
- 손실이 감소하면 학습이 진행 중

**pipeline · Trainer의 범위**

- 표준적인 입출력 구조에는 편리
- 특수한 구조는 모델을 직접 다룸

In [ ]:
model.train()
t0 = time.time()

for epoch in range(EPOCHS):
    losses = []

    for i, batch in enumerate(train_loader):
        outputs = model(
            pixel_values=batch["pixel_values"].to(device),
            input_boxes=batch["input_boxes"].to(device),
            multimask_output=False,
        )

        pred = outputs.pred_masks.squeeze(1)
        gt = batch["ground_truth_mask"].unsqueeze(1).to(device)

        loss = loss_fn(pred, gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if i % 50 == 0:
            print(f"  epoch {epoch+1} step {i} — loss {loss.item():.4f}",
                  flush=True)

    print(f"[epoch {epoch+1}] 평균 손실 {np.mean(losses):.4f}", flush=True)

print("총 소요 시간:", round((time.time() - t0) / 60, 1), "분")

---
## 6. 결과 확인

### 6-1. 예측·평가 함수

- `predict_mask` — 이미지와 박스로 마스크 생성
- `dice_score` — 예측과 정답의 겹치는 정도 (1에 가까울수록 정확)
- 평가 시 박스는 흔들지 않음 — 공정한 비교

In [ ]:
def predict_mask(m, image, box):
    """이미지와 박스로 마스크 예측"""
    m.eval()
    inputs = processor(
        image.convert("RGB"), input_boxes=[[box]], return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        out = m(**inputs, multimask_output=False)

    prob = torch.sigmoid(out.pred_masks.squeeze())
    return (prob > 0.5).cpu().numpy().astype(np.uint8) * 255


def dice_score(pred, gt_mask):
    """예측과 정답의 일치도 (0~1)"""
    p = (np.array(Image.fromarray(pred).resize((256, 256))) > 0).astype(np.float32)
    g = (np.array(gt_mask.convert("L").resize((256, 256))) > 0).astype(np.float32)

    return 2 * (p * g).sum() / (p.sum() + g.sum() + 1e-8)


print("함수 준비 완료")

### 6-2. 비교용 원본 모델

- 미세조정하지 않은 SAM을 별도로 불러옴
- 같은 이미지 · 같은 박스로 두 모델을 비교

In [ ]:
base_model = SamModel.from_pretrained("facebook/sam-vit-base").to(device)

print("원본 모델 준비 완료")

### 6-3. 시각 비교

- 원본 · 정답 · 미세조정 전 · 미세조정 후를 나란히 표시
- 실행할 때마다 **다른 이미지** 선택
- 같은 이미지를 반복하려면 `seed` 지정

In [ ]:
def show_row(images, titles, size=(220, 220)):
    """이미지 여러 장을 가로로 배치"""
    imgs = [im.convert("RGB").resize(size) for im in images]
    w, h = size
    canvas = Image.new("RGB", (w * len(imgs), h), "white")
    for i, im in enumerate(imgs):
        canvas.paste(im, (i * w, 0))
    return canvas


def show_comparison(n=3, seed=None):
    """무작위 n장에 대해 전후 비교"""
    if seed is not None:
        random.seed(seed)

    print("원본 | 정답 | 미세조정 전 | 미세조정 후\n")

    for idx in random.sample(range(len(ds_test)), n):
        it = ds_test[idx]
        gt = it[MASK_COL]
        box = get_bounding_box(gt, shift=0)

        before = predict_mask(base_model, it["image"], box)
        after = predict_mask(model, it["image"], box)

        print(f"[{idx}] Dice — 전 {dice_score(before, gt):.3f} "
              f"/ 후 {dice_score(after, gt):.3f}")

        display(show_row(
            [it["image"], gt,
             Image.fromarray(before), Image.fromarray(after)],
            ["원본", "정답", "전", "후"],
        ))
        print()


show_comparison(n=3)

### 6-4. 평균 성능 비교

- 한 장만으로는 판단 어려움
- 여러 장의 평균 Dice로 개선 여부 확인

In [ ]:
n_eval = 100
before_scores, after_scores = [], []

for i in range(min(n_eval, len(ds_test))):
    it = ds_test[i]
    gt = it[MASK_COL]
    box = get_bounding_box(gt, shift=0)

    before_scores.append(dice_score(predict_mask(base_model, it["image"], box), gt))
    after_scores.append(dice_score(predict_mask(model, it["image"], box), gt))

print(f"미세조정 전 평균 Dice: {np.mean(before_scores):.4f}")
print(f"미세조정 후 평균 Dice: {np.mean(after_scores):.4f}")
print(f"변화: {np.mean(after_scores) - np.mean(before_scores):+.4f}")

**결과 해석**

- 값이 올랐으면 — 미세조정이 효과를 보임
- 값이 그대로거나 낮아졌으면 — 학습이 부족하거나 설정이 부적절

**조정할 수 있는 항목**

- `N_TRAIN` — 학습 이미지 수 늘리기
- `EPOCHS` — 반복 횟수 늘리기
- `LR` — 학습률 조정
- `dice_weight` — 손실 함수의 비중 조정

**학습을 수행했다고 해서 항상 개선되는 것은 아니다.** 전후 비교가 필요한 이유이다.

---
## 7. 모델 저장

- 모델과 처리기를 함께 저장
- 이후 추론에 바로 사용 가능

In [ ]:
model.save_pretrained("sam_polyp")
processor.save_pretrained("sam_polyp")

import os
print(os.listdir("sam_polyp"))

---
## 8. 정리

**수행 내용**

- 사전학습 SAM의 인코더를 동결하고 마스크 디코더만 학습
- 학습 대상 — 전체의 약 4.3%
- Dice와 BCE를 결합한 손실 함수 사용
- 전후 성능을 동일 기준으로 비교

**앞선 실습과의 연결**

- 추론 실습 — SAM은 경계를 찾으나 대상을 판별하지 못함
- 이번 실습 — 의료 데이터로 조정하여 특정 대상 분할
- 도입부 연구 사례(H-SAM · MedSAM)와 동일한 방식

**확인한 사항**

- 일부 파라미터만 학습해도 도메인 적응 가능
- 손실 함수 선택이 결과를 좌우 — 대상이 작을수록 Dice가 중요
- 학습량이 부족하면 개선이 미미하거나 오히려 저하

**pipeline · Trainer의 범위**

- 표준적인 입출력에는 편리한 도구
- 입력이 특수한 경우(이미지 + 박스 + 마스크)는 **직접 작성**
- 모델 카드의 사용 예제를 확인하는 습관 필요

---

데이터 출처: kowndinya23/Kvasir-SEG (Hugging Face Hub 공개)
참고: Ma et al., "Segment Anything in Medical Images", Nature Communications (2024)